# Customer Segmentation and Churn Analysis — Telecom

**Objective:** Explore recorded churn in the IBM Telco Customer Churn sample, identify associated customer and account characteristics, build transparent customer segments, and recommend retention actions.

**Required coverage:** EDA, boxplots, heatmap/correlation analysis, crosstabs, insight generation, and segmentation.

> This is a descriptive coursework analysis. Associations do not establish causation, and this is not a production churn prediction model.

## 1. Setup and imports

Install dependencies from the project root before running this notebook:

```bash
python -m pip install -r requirements.txt
```

The project downloads and verifies the original IBM CSV if it is not already present in `data/raw/`.

In [ ]:
from pathlib import Path
import sys

# Make the project package importable whether Jupyter starts in the root or notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from IPython.display import display, Image, Markdown

from src.analysis import (
    load_raw_data,
    clean_and_engineer,
    assign_customer_segments,
    categorical_churn_table,
    correlation_features,
    segment_summary,
    retention_action,
    make_eda_figures,
    write_outputs,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")


## 2. Load and inspect the source data

The raw sample contains customer-level account, service, billing, and churn fields. `customerID` is an identifier and is not used as a numeric explanatory variable.

In [ ]:
raw_df = load_raw_data()
print(f"Dataset shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
display(raw_df.head())
raw_df.info()


## 3. Data quality and preparation

`TotalCharges` is stored as text in the source. Blank values for zero-tenure customers are treated as zero because no tenure has accrued yet. The target is encoded as `churn_flag` (`No = 0`, `Yes = 1`). The engineered `service_count` counts subscribed services with an explicit `Yes` value.

In [ ]:
df = clean_and_engineer(raw_df)
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
})
display(quality)
print(f"Duplicate customer IDs: {df['customerID'].duplicated().sum():,}")
print(f"Churn rate: {df['churn_flag'].mean():.1%}")


## 4. Overall churn distribution

Start with the target balance before comparing customer attributes.

In [ ]:
churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"])
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=churn_counts.index, y=churn_counts.values, hue=churn_counts.index, legend=False, ax=ax)
ax.set(title="Customer churn distribution", xlabel="Churn", ylabel="Customers")
for patch, value in zip(ax.patches, churn_counts.values):
    ax.annotate(f"{value:,}", (patch.get_x() + patch.get_width()/2, patch.get_height()),
                ha="center", va="bottom", xytext=(0, 4), textcoords="offset points")
plt.show()


## 5. Boxplots: tenure and charges by churn

The boxplots compare the distributions of tenure, monthly charges, and accumulated charges for customers who churned and those who did not. Inspect medians, spread, overlap, and outliers; do not interpret these plots as causal evidence.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, feature in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.boxplot(data=df, x="Churn", y=feature, hue="Churn", legend=False, ax=ax)
    ax.set_title(f"{feature} by churn")
    ax.set_xlabel("Churn")
    ax.set_ylabel(feature)
plt.tight_layout()
plt.show()


## 6. Correlation analysis and heatmap

The correlation matrix includes numeric features and one-hot indicators for `Contract`. This preserves the nominal meaning of contract type instead of assigning arbitrary ordinal codes. The binary churn flag is included to show linear associations with numeric / indicator variables. Payment method, internet service, and tech support are also compared with churn rates in the crosstabs below.

In [ ]:
# Contract is categorical, so include one-hot indicator columns instead of assigning it ordinal codes.
corr_frame = correlation_features(df)
corr = corr_frame.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, ax=ax)
ax.set_title("Correlation heatmap")
plt.tight_layout()
plt.show()
display(corr["churn_flag"].drop("churn_flag").sort_values())


## 7. Crosstabs and churn rates by category

For each category, churn rate is calculated as churned customers divided by customers in that category. The count column is important: a high rate in a very small group may represent fewer customers than a moderate rate in a large group.

In [ ]:
category_features = ["Contract", "tenure_group", "PaymentMethod", "InternetService", "TechSupport"]
for feature in category_features:
    print(f"\nChurn by {feature}")
    display(categorical_churn_table(df, feature).round(3))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, feature in zip(axes.flat, ["Contract", "PaymentMethod", "InternetService", "TechSupport"]):
    rates = categorical_churn_table(df, feature).sort_values("churn_rate")
    sns.barplot(data=rates, x="churn_rate_pct", y=feature, hue=feature, legend=False, ax=ax)
    ax.set(title=f"Churn rate by {feature}", xlabel="Churn rate (%)", ylabel="")
    ax.set_xlim(0, max(10, rates["churn_rate_pct"].max() * 1.15))
plt.tight_layout()
plt.show()


## 8. Rule-based customer segmentation

The segment rules are deliberately interpretable and do **not** use the churn outcome. They are applied in priority order:

1. **At-risk:** tenure of 12 months or less, or month-to-month with monthly charges at/above the sample's 75th percentile or no tech support.
2. **Loyal:** tenure of at least 48 months and a one- or two-year contract, unless already assigned At-risk.
3. **High-value:** monthly charges at/above the 75th percentile, unless already assigned At-risk or Loyal.
4. **Core:** all remaining customers.

Churn rates are measured only after assigning the segments, so the target does not leak into the grouping rules.

In [ ]:
segmented_df, value_threshold = assign_customer_segments(df)
print(f"75th percentile monthly-charge threshold: {value_threshold:.2f} dataset currency units")
seg_summary = segment_summary(segmented_df)
display(seg_summary.style.format({
    "share_of_customers": "{:.1%}",
    "avg_monthly_charges": "{:.2f}",
    "churn_rate": "{:.1%}",
}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=seg_summary, x="customer_segment", y="customers", hue="customer_segment", legend=False, ax=axes[0])
axes[0].set(title="Customer count by segment", xlabel="", ylabel="Customers")
sns.barplot(data=seg_summary, x="customer_segment", y="churn_rate", hue="customer_segment", legend=False, ax=axes[1])
axes[1].set(title="Historical churn rate by segment", xlabel="", ylabel="Churn rate")
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0%}"))
plt.tight_layout()
plt.show()


## 9. Retention recommendations

Use the segment-level results to prioritise the type of intervention. The recommendations below are starting hypotheses; validate them with controlled experiments and monitor both retention and margin.

In [ ]:
for segment in ["At-risk", "High-value", "Loyal", "Core"]:
    n = int(segmented_df["customer_segment"].eq(segment).sum())
    print(f"{segment} ({n:,} customers): {retention_action(segment)}\n")


## 10. Export analysis artifacts

This step writes a generated Markdown summary, tables, and figure files into `reports/`. It is also available from the command line with `python -m src.run_analysis`.

In [ ]:
outputs = write_outputs(segmented_df, value_threshold, PROJECT_ROOT / "reports")
for name, path in outputs.items():
    print(f"{name}: {path}")

summary_path = PROJECT_ROOT / "reports" / "analysis_summary.md"
print("\nGenerated analysis summary:")
print(summary_path.read_text(encoding="utf-8"))


## 11. Conclusions and interpretation limits

After running all cells, summarise the strongest observed churn differences from the category tables and boxplots. The generated `reports/analysis_summary.md` records the dataset-specific figures automatically.

When writing the final submission, make sure to:

- State the overall churn rate and dataset size.
- Compare contract types and tenure bands using both churn rates and group counts.
- Discuss tenure, monthly charges, and total charges from the boxplots and correlation heatmap.
- Identify which segment has the largest historical churn rate and how many customers it contains.
- Link each recommendation to the pattern it is intended to address.
- State clearly that the dataset is a cross-sectional sample and correlation does not prove causation.